# `rag.search` Tool: Real Test

Covers task backlog §3.2. Same pattern as
[3.1 server skeleton smoke test.ipynb](3.1%20server%20skeleton%20smoke%20test.ipynb): a real
subprocess spawn of `mcp_server.py`, talking over the real stdio transport via the MCP
client SDK -- not an in-process shortcut, not a fixture/mocked index. `rag_search_tool.py`
wraps §2.4's `retriever.py` `search()` directly against the real, committed
`products_home_kitchen` Chroma index -- no separate fixture index was built (the real
index is already small, fast, and committed, so a synthetic fixture wouldn't add anything
the real one doesn't already give for free).

In [1]:
# !pip install mcp

## Setup

In [2]:
import os
import sys

from mcp.client.session import ClientSession
from mcp.client.stdio import StdioServerParameters, stdio_client

SERVER_PATH = os.path.join(os.getcwd(), "mcp_server.py")
server_params = StdioServerParameters(command=sys.executable, args=[SERVER_PATH])
print(f"Will spawn: {sys.executable} {SERVER_PATH}")

Will spawn: /opt/anaconda3/bin/python /Users/pareeyaporn/Desktop/ADSP 32028 Applied Generative AI - Agents and Multimodal Intelligence/Project/Code/3. MCP Server & Tools/mcp_server.py


## Real calls through the real MCP server

In [3]:
import json


async def call_rag_search(session, **kwargs):
    result = await session.call_tool("rag.search", kwargs)
    assert not result.is_error, f"unexpected tool error: {result.content[0].text}"
    return json.loads(result.content[0].text)


async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()

        print("-- Case 1: plain query --")
        r1 = await call_rag_search(session, query="insulated lunch box for kids", top_k=3)
        for p in r1["products"]:
            print(f"  [{p['distance']:.3f}] {p['title'][:60]}  ${p['price']:.2f}  doc_id={p['doc_id'][:8]}...")
        assert len(r1["products"]) == 3
        assert r1["warnings"] == []
        assert set(r1["products"][0]) == {
            "doc_id", "title", "price", "price_per_unit", "price_unit", "brand", "category_top", "url", "distance",
        }

        print("\n-- Case 2: budget_max filter --")
        r2 = await call_rag_search(session, query="insulated lunch box for kids", top_k=5, budget_max=15)
        print(f"  {len(r2['products'])} products, all <= $15: {all(p['price'] <= 15 for p in r2['products'])}")
        assert all(p["price"] <= 15 for p in r2["products"])

        print("\n-- Case 3: brand filter --")
        r3 = await call_rag_search(session, query="insulated lunch box", brand="Wildkin", top_k=5)
        print(f"  {len(r3['products'])} products, all Wildkin: {all('wildkin' in (p['brand'] or '').lower() for p in r3['products'])}")
        assert all("wildkin" in (p["brand"] or "").lower() for p in r3["products"])

        print("\n-- Case 4: category filter -- should warn, not silently no-op --")
        r4 = await call_rag_search(session, query="kids bedding set", category="Bedding")
        print(f"  warnings: {r4['warnings']}")
        assert len(r4["warnings"]) == 1 and "category" in r4["warnings"][0]

        print("\n[OK] all 4 cases passed against the real index, through the real MCP server.")

-- Case 1: plain query --
  [0.589] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect  $12.77  doc_id=7c3f2bf5...
  [0.649] Wildkin Kids Insulated Lunch Box for Boys and Girls, Perfect  $10.88  doc_id=5d953e35...
  [0.687] Wildkin 33600 Monsters Lunch Box, Insulated, Moisture Resist  $14.99  doc_id=b1904b27...

-- Case 2: budget_max filter --
  5 products, all <= $15: True

-- Case 3: brand filter --
  5 products, all Wildkin: True

-- Case 4: category filter -- should warn, not silently no-op --
  warnings: ["category filter not applied: category_top is constant across products_home_kitchen (§2.4 finding), so it can't narrow results here"]

[OK] all 4 cases passed against the real index, through the real MCP server.


## Next steps

- §3.3: implement `web.search` for real (Tavily)
- §3.4: structured logging, secret scrubbing, typed error contract, MCP README
- Open follow-up (not decided here): whether §2.4's `retriever_node` should swap from
  calling `search()` directly to calling this tool over MCP -- a real architecture change
  (sync LangGraph node becoming an async MCP client), not done as a side effect of §3.2